# 03_00 — Model Training: Feature Selection

**Pipeline:** `model_training`  
**Kedro nodes:** `build_training_data_node` → `train_isolation_forest_node` → `evaluate_model_node`  
**Input catalog:** `prepared_flights` (data/03_primary/)  
**Output catalog:** `isolation_forest`, `model_metrics`

---

## Contexto

Com os dados preparados, esta etapa treina um detector de anomalias não-supervisionado.
O objetivo é detectar a **falha do motor** o mais rápido possível após ela ocorrer.

### Por que Isolation Forest?

- Não requer dados rotulados para treinar (apenas o parâmetro `contamination`)
- Funciona bem com dados de alta dimensão e temporal
- Produz um **anomaly score** contínuo, não só uma classificação binária

### Fluxo da pipeline
1. **Sliding windows**: cada amostra = últimos 20 timesteps achatados (0.2s de contexto temporal)
2. **Seleção de features**: Random Forest seleciona as top 10 por importância
3. **Treino**: Isolation Forest com split temporal 70/30
4. **Avaliação**: latência de detecção = tempo entre falha real e primeiro alerta

> **Para rodar toda a pipeline no Kedro:** `kedro run --pipeline=model_training`

## Imports e parâmetros

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Funcoes da pipeline — as mesmas que o Kedro executa
from aeroespacial_2.pipelines.model_training.nodes import (
    create_windows,
    evaluate_model,
    select_top_features_rf,
    split_windows,
    train_isolation_forest,
)

# Parâmetros espelhando conf/base/parameters.yml
PREPARED_FILE = "../data/04_feature/carbonZ_2018-07-18-15-53-31_1_engine_failure.csv"
WINDOW_SIZE = 20
N_TOP_FEATURES = 10
CONTAMINATION = 0.001  # baixo para baseline
N_ESTIMATORS = 100
TRAIN_RATIO = 0.7
TARGET_COL = "target_fault"
TIMESTAMP_COL = "timestamp"

In [ ]:
df = pd.read_csv(PREPARED_FILE)
print(f"Shape: {df.shape}")
print(f"Falhas: {df[TARGET_COL].sum():.0f} amostras ({df[TARGET_COL].mean()*100:.1f}%)")
df.head(3)

## Passo 1 — Seleção de features com Random Forest

Usamos um Random Forest **supervisionado** como ranqueador de features.
Ele explora os rótulos para identificar quais sinais mudam mais
discriminativamente no momento da falha.

Isso reduz a dimensão do sliding window e melhora a detecção do Isolation Forest.

In [ ]:
features = [c for c in df.columns if c not in [TARGET_COL, TIMESTAMP_COL]]
top_features = select_top_features_rf(df, features, TARGET_COL, n_top=N_TOP_FEATURES)

print(f"Top {N_TOP_FEATURES} features:")
for i, f in enumerate(top_features, 1):
    print(f"  {i:2d}. {f}")

## Passo 2 — Sliding windows

Cada amostra é o achatamento dos últimos `window_size` timesteps.
A 100 Hz, `window_size=20` captura 0.2 segundos de contexto temporal.

In [ ]:
X, y = create_windows(df, WINDOW_SIZE, top_features, TARGET_COL)
print(f"X shape: {X.shape}  (n_amostras, window_size × n_features)")
print(f"y shape: {y.shape}")
print(f"Positivos (falha): {y.sum()} ({y.mean()*100:.1f}%)")

## Passo 3 — Split temporal e treino

O split é **temporal** (sem shuffle): os primeiros 70% para treino, os últimos 30% para teste.
O embaralhamento vazaria informações sobre a falha futura para o treino.

In [ ]:
X_train, X_test, y_train, y_test = split_windows(X, y, train_ratio=TRAIN_RATIO)
timestamps = df[TIMESTAMP_COL].values[WINDOW_SIZE:]
split_idx = int(len(timestamps) * TRAIN_RATIO)
ts_test = timestamps[split_idx:]

print(f"Train: {X_train.shape} | {y_train.sum():.0f} falhas")
print(f"Test:  {X_test.shape}  | {y_test.sum():.0f} falhas")

In [ ]:
model = train_isolation_forest(X_train, contamination=CONTAMINATION, n_estimators=N_ESTIMATORS)
print("Modelo treinado!")

## Passo 4 — Avaliação

In [ ]:
metrics = evaluate_model(model, X_test, y_test, ts_test)
print(f"Falha real:     {metrics.get('real_fault_time_s', 'N/A'):.2f}s")
print(f"Detecao:        {metrics.get('predicted_fault_time_s', 'N/A'):.2f}s")
print(f"Latencia:       {metrics.get('detection_latency_s', 'N/A'):.2f}s")

In [ ]:
# Predicoes e scores
raw_preds = model.predict(X_test)
y_pred = np.where(raw_preds == -1, 1, 0)
scores = model.decision_function(X_test)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Painel 1: falha real vs detectada
axes[0].plot(ts_test, y_test, color="red", alpha=0.6, linewidth=2, label="Falha real")
axes[0].plot(ts_test, y_pred, color="blue", linestyle="--", linewidth=1, label="Detecção")
axes[0].set_ylabel("Estado (0=normal, 1=falha)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title("Isolation Forest — Falha Real vs Detecção")

# Painel 2: anomaly score
axes[1].plot(ts_test, scores, color="purple", linewidth=0.8, label="Anomaly Score")
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5, label="Limiar")
if metrics.get("real_fault_time_s"):
    axes[1].axvline(metrics["real_fault_time_s"], color="black", linewidth=1.5, label="Falha real")
axes[1].set_ylabel("Score (negativo = mais anômalo)")
axes[1].set_xlabel("Tempo (s)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_title('"Quão suspeito" o modelo acha o voo')

plt.tight_layout()
plt.show()

---
**Próximo:** `03_01_model2.ipynb` → tuning do parâmetro `contamination` para otimizar a latência de detecção.